In [10]:
import datetime
import io
import sys
import os
from pathlib import Path
import unicodedata
from datetime import timezone

import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import pybaseball
import requests
import scipy.stats as stats
from tqdm import tqdm
from bs4 import BeautifulSoup
from pybaseball import statcast
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor, XGBClassifier
from sklearn.calibration import calibration_curve

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'mlb_props').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from somewhere inside the MLB-Props repository.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from mlb_props.config import PITCHER_STARTS_PATH, ROSTER_SCRAPER_DIR
from mlb_props.features import TARGET, validate_pregame_features

sys.path.insert(0, str(ROSTER_SCRAPER_DIR))
from RosterScraper import RosterScraper

In [11]:
if not PITCHER_STARTS_PATH.exists():
    raise FileNotFoundError(
        f"Pitcher-start data not found at {PITCHER_STARTS_PATH}. "
        "Set MLB_PROPS_PITCHER_STARTS or MLB_PROPS_DATA_DIR."
    )
Starts = pl.read_parquet(PITCHER_STARTS_PATH)
df = Starts.to_pandas()

In [12]:
df = df.sort_values(['player_name', 'game_date']).reset_index(drop=True)

# ── Derived columns ───────────────────────────────────────────────────────────
df['k_rate'] = df['K'] / df['PA'].clip(lower=1)
df['p_throws_enc'] = (df['p_throws'] == 'R').astype(int)

# ── Rolling features (lagged by 1 game) ──────────────────────────────────────
roll_features = {}
for window in tqdm([5, 10, 20], desc='Rolling windows'):
    for stat in ['k_rate', 'Whiffs', 'CSW', 'CS', 'GB', 'BB', 'Strikes', 'Balls', 'BIP']:
        col = f'{stat}_P{window}'
        roll_features[col] = (
            df.groupby('player_name')[stat]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=3).mean())
        )
df = pd.concat([df, pd.DataFrame(roll_features)], axis=1)

# ── Interaction features (Lasso-informed) ─────────────────────────────────────
df['ff_power_angle']   = df['ff_velo'] * df['ff_vaa'].abs()
df['ch_effectiveness'] = df['throws_ch'] * df['ch_vaa'].abs()

# ── Feature definitions ───────────────────────────────────────────────────────
physics_features = [
    'ff_velo', 'ff_spinrate', 'ff_ivb', 'ff_hb', 'ff_vaa',
    'sl_velo', 'sl_spinrate', 'sl_ivb', 'sl_hb', 'sl_vaa',
    'ch_velo', 'ch_spinrate', 'ch_ivb', 'ch_hb', 'ch_vaa',
    'si_velo', 'si_spinrate', 'si_ivb', 'si_hb', 'si_vaa',
    'cu_velo', 'cu_spinrate', 'cu_ivb', 'cu_hb', 'cu_vaa',
    'fc_velo', 'fc_spinrate', 'fc_ivb', 'fc_hb', 'fc_vaa',
    'st_velo', 'st_spinrate', 'st_ivb', 'st_hb', 'st_vaa',
]

# P20 only — Lasso zeroed out all P5 and P10 features
rolling_features = [
    c for c in df.columns
    if any(c.startswith(s) for s in ['k_rate_P', 'Whiffs_P', 'CSW_P', 'CS_P',
                                      'GB_P', 'BB_P', 'Strikes_P', 'Balls_P', 'BIP_P'])
    and c.endswith('_P20')
]

arsenal_features = [
    'throws_ff', 'throws_si', 'throws_fc', 'throws_sl',
    'throws_st', 'throws_cu', 'throws_ch',
    'ff_usage_vR', 'ff_usage_vL',
    'sl_usage_vR', 'sl_usage_vL',
    'ch_usage_vR', 'ch_usage_vL',
]
# Same-game PA is used to define the target, never as a model input.
context_features = []

ALL_FEATURES = list(validate_pregame_features(
    physics_features + rolling_features +
    arsenal_features + context_features +
    ['p_throws_enc', 'ff_power_angle', 'ch_effectiveness']
))
print(f"Total features: {len(ALL_FEATURES)}")

# ── Impute nulls ──────────────────────────────────────────────────────────────
for col in physics_features + ['ff_power_angle', 'ch_effectiveness']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

for col in rolling_features:
    df[col] = df[col].fillna(0)

Rolling windows: 100%|██████████| 3/3 [00:04<00:00,  1.62s/it]

Total features: 61


In [ ]:
# ── Chronological 70/15/15 split ─────────────────────────────────────────────
model_df = df[ALL_FEATURES + [TARGET, 'game_date']].dropna()
print(f"Rows after imputation: {len(model_df)}")

dates = model_df['game_date'].sort_values()
n = len(model_df)
train_cutoff = dates.iloc[int(n * 0.70)]
val_cutoff   = dates.iloc[int(n * 0.85)]

train = model_df[model_df['game_date'] <  train_cutoff]
val   = model_df[(model_df['game_date'] >= train_cutoff) &
                 (model_df['game_date'] <  val_cutoff)]
test  = model_df[model_df['game_date'] >= val_cutoff]

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"Train ends:  {train['game_date'].max()}")
print(f"Val ends:    {val['game_date'].max()}")
print(f"Test starts: {test['game_date'].min()}")

X_train = train[ALL_FEATURES]
y_train = train[TARGET]
X_val   = val[ALL_FEATURES]
y_val   = val[TARGET]
X_test  = test[ALL_FEATURES]
y_test  = test[TARGET]

# ── Baseline carried from linear model (for comparison) ──────────────────────
BASELINE_VAL_R2  = 0.1274   # Ridge alpha=10
BASELINE_TEST_R2 = 0.1486   # Linear Regression / Ridge alpha=1
print(f"\nBaseline to beat — Val R²: {BASELINE_VAL_R2}, Test R²: {BASELINE_TEST_R2}")

# ── Leakage check ─────────────────────────────────────────────────────────────
leak_check = pd.DataFrame({
    'feature': ALL_FEATURES,
    'same_game_corr': [
        model_df[f].corr(model_df[TARGET])
        for f in tqdm(ALL_FEATURES, desc='Leakage check')
    ]
}).sort_values('same_game_corr', ascending=False)
print("\nTop 10 same-game correlations:")
print(leak_check.head(10).to_string(index=False))

# ── Naive baseline ────────────────────────────────────────────────────────────
naive = DummyRegressor(strategy='mean')
naive.fit(X_train, y_train)

for name, X, y in [('Val', X_val, y_val), ('Test', X_test, y_test)]:
    preds = naive.predict(X)
    print(f"\nNaive Mean [{name}]")
    print(f"  RMSE: {np.sqrt(mean_squared_error(y, preds)):.4f}")
    print(f"  MAE:  {mean_absolute_error(y, preds):.4f}")
    print(f"  R²:   {r2_score(y, preds):.4f}")

# ── Linear Regression ─────────────────────────────────────────────────────────
lr_pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
lr_pipe.fit(X_train, y_train)

for name, X, y in [('Val', X_val, y_val), ('Test', X_test, y_test)]:
    preds = lr_pipe.predict(X)
    print(f"\nLinear Regression [{name}]")
    print(f"  RMSE: {np.sqrt(mean_squared_error(y, preds)):.4f}")
    print(f"  MAE:  {mean_absolute_error(y, preds):.4f}")
    print(f"  R²:   {r2_score(y, preds):.4f}")

# ── Ridge Regression ──────────────────────────────────────────────────────────
ridge_pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))])
ridge_pipe.fit(X_train, y_train)

for name, X, y in [('Val', X_val, y_val), ('Test', X_test, y_test)]:
    preds = ridge_pipe.predict(X)
    print(f"\nRidge Regression (alpha=10) [{name}]")
    print(f"  RMSE: {np.sqrt(mean_squared_error(y, preds)):.4f}")
    print(f"  MAE:  {mean_absolute_error(y, preds):.4f}")
    print(f"  R²:   {r2_score(y, preds):.4f}")

# ── Ridge alpha sweep ─────────────────────────────────────────────────────────
alphas = [0.01, 0.1, 1, 10, 50, 100, 500]
print(f"\n{'Alpha':>8} {'Val R²':>10} {'Test R²':>10}")
for alpha in tqdm(alphas, desc='Ridge alpha sweep'):
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=alpha))])
    pipe.fit(X_train, y_train)
    val_r2  = r2_score(y_val,  pipe.predict(X_val))
    test_r2 = r2_score(y_test, pipe.predict(X_test))
    print(f"{alpha:>8} {val_r2:>10.4f} {test_r2:>10.4f}")

# ── Ridge coefficients ────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'feature': X_train.columns.tolist(),
    'coef': ridge_pipe.named_steps['model'].coef_
})
coef_df['abs_coef'] = coef_df['coef'].abs()
print("\nTop 15 Ridge coefficients:")
print(coef_df.sort_values('abs_coef', ascending=False).head(15).to_string(index=False))

# ── Lasso alpha sweep ─────────────────────────────────────────────────────────
lasso_alphas = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1]
print(f"\n{'Lasso α':>10} {'Val R²':>10} {'Test R²':>10} {'Nonzero coefs':>15}")
for alpha in tqdm(lasso_alphas, desc='Lasso alpha sweep'):
    pipe = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=alpha, max_iter=10000))])
    pipe.fit(X_train, y_train)
    val_r2  = r2_score(y_val,  pipe.predict(X_val))
    test_r2 = r2_score(y_test, pipe.predict(X_test))
    nonzero = np.sum(pipe.named_steps['model'].coef_ != 0)
    print(f"{alpha:>10} {val_r2:>10.4f} {test_r2:>10.4f} {nonzero:>15}")

# ── Elastic Net sweep ─────────────────────────────────────────────────────────
enet_configs = [(0.001, 0.5), (0.001, 0.7), (0.001, 0.9),
                (0.005, 0.5), (0.005, 0.7), (0.005, 0.9)]
print(f"\n{'α':>8} {'l1_ratio':>10} {'Val R²':>10} {'Test R²':>10} {'Nonzero':>10}")
for alpha, l1_ratio in tqdm(enet_configs, desc='ElasticNet sweep'):
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('model', ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000))])
    pipe.fit(X_train, y_train)
    val_r2  = r2_score(y_val,  pipe.predict(X_val))
    test_r2 = r2_score(y_test, pipe.predict(X_test))
    nonzero = np.sum(pipe.named_steps['model'].coef_ != 0)
    print(f"{alpha:>8} {l1_ratio:>10} {val_r2:>10.4f} {test_r2:>10.4f} {nonzero:>10}")

# ── Lasso feature selection output ───────────────────────────────────────────
best_lasso = Pipeline([('scaler', StandardScaler()),
                       ('model', Lasso(alpha=0.001, max_iter=10000))])
best_lasso.fit(X_train, y_train)

lasso_coef_df = pd.DataFrame({
    'feature': X_train.columns.tolist(),
    'coef': best_lasso.named_steps['model'].coef_
})
lasso_coef_df['abs_coef'] = lasso_coef_df['coef'].abs()

print("\nLasso — features kept (nonzero coefficients):")
print(lasso_coef_df[lasso_coef_df['coef'] != 0]
      .sort_values('abs_coef', ascending=False).to_string(index=False))

print("\nLasso — features zeroed out:")
print(lasso_coef_df[lasso_coef_df['coef'] == 0]['feature'].tolist())

Rows after imputation: 14352
Train: 10,046 | Val: 2,128 | Test: 2,178
Train ends:  2025-04-15 00:00:00
Val ends:    2025-07-05 00:00:00
Test starts: 2025-07-06 00:00:00

Naive Mean [Val]
  RMSE: 0.1071
  MAE:  0.0857
  R²:   -0.0031

Naive Mean [Test]
  RMSE: 0.1098
  MAE:  0.0868
  R²:   -0.0001

Linear Regression [Val]
  RMSE: 0.1000
  MAE:  0.0790
  R²:   0.1263

Linear Regression [Test]
  RMSE: 0.1014
  MAE:  0.0800
  R²:   0.1485

Ridge Regression (alpha=10) [Val]
  RMSE: 0.0999
  MAE:  0.0789
  R²:   0.1274

Ridge Regression (alpha=10) [Test]
  RMSE: 0.1014
  MAE:  0.0800
  R²:   0.1475

   Alpha     Val R²    Test R²
    0.01     0.1263     0.1485
     0.1     0.1263     0.1486
       1     0.1266     0.1486
      10     0.1274     0.1475
      50     0.1274     0.1440
     100     0.1267     0.1418
     500     0.1257     0.1383

Top 15 Ridge coefficients:
    feature      coef  abs_coef
    ff_velo  0.115607  0.115607
  throws_ff -0.114092  0.114092
  throws_ch -0.035778  0.03